# Jonathan Elahee
# Jennifer Lim
# Palwinder Singh

# Assignment 3: Decision Tree Root Node Splitter
## Pima Indians Diabetes Dataset

This notebook implements a decision tree root node splitter using:
- Gini Index as impurity measure
- Entropy as impurity measure

The goal is to find the best feature and threshold for splitting the root node.

In [12]:
import numpy as np
import pandas as pd

## 1. Load and Explore Dataset

In [13]:
df = pd.read_csv('diabetes.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nClass distribution:")
print(df['Outcome'].value_counts())

Dataset shape: (768, 9)

First few rows:
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  

Class distribution:
Outcome
0    500
1    268
Name: count, dtype: int64


In [14]:
# Separate features and target
X = df.drop('Outcome', axis=1).values
y = df['Outcome'].values
feature_names = df.drop('Outcome', axis=1).columns.tolist()

print(f"Features: {feature_names}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

Features: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
X shape: (768, 8)
y shape: (768,)


## 2. Implement Impurity Measures

In [15]:
def gini_index(y):
    """Calculate Gini impurity for a set of labels"""
    if len(y) == 0:
        return 0
    classes, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    return 1 - np.sum(probabilities ** 2)

def entropy(y):
    """Calculate entropy for a set of labels"""
    if len(y) == 0:
        return 0
    classes, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    probabilities = probabilities[probabilities > 0]
    return -np.sum(probabilities * np.log2(probabilities))

In [16]:
print(f"Initial Gini Index: {gini_index(y):.4f}")
print(f"Initial Entropy: {entropy(y):.4f}")

Initial Gini Index: 0.4544
Initial Entropy: 0.9331


## 3. Implement Split Quality Calculation

In [17]:
def weighted_impurity(y_left, y_right, impurity_func):
    """Calculate weighted impurity after split"""
    n = len(y_left) + len(y_right)
    if n == 0:
        return 0
    p_left = len(y_left) / n
    p_right = len(y_right) / n
    return p_left * impurity_func(y_left) + p_right * impurity_func(y_right)

def information_gain(y_parent, y_left, y_right, impurity_func):
    """Calculate information gain from a split"""
    parent_impurity = impurity_func(y_parent)
    children_impurity = weighted_impurity(y_left, y_right, impurity_func)
    return parent_impurity - children_impurity

## 4. Find Best Split for a Feature

In [18]:
def find_best_split_for_feature(X_feature, y, impurity_func):
    """Find best threshold for splitting on a single feature"""
    unique_values = np.unique(X_feature)
    
    if len(unique_values) == 1:
        return None, -np.inf
    
    # Calculate midpoints between consecutive unique values
    thresholds = [(unique_values[i] + unique_values[i+1]) / 2 
                  for i in range(len(unique_values) - 1)]
    
    best_threshold = None
    best_gain = -np.inf
    
    for threshold in thresholds:
        left_mask = X_feature <= threshold
        right_mask = ~left_mask
        
        y_left = y[left_mask]
        y_right = y[right_mask]
        
        if len(y_left) == 0 or len(y_right) == 0:
            continue
        
        gain = information_gain(y, y_left, y_right, impurity_func)
        
        if gain > best_gain:
            best_gain = gain
            best_threshold = threshold
    
    return best_threshold, best_gain

## 5. Find Best Split for All Features Using Gini Index

In [23]:
print("="*70)
print("BEST SPLITS USING GINI INDEX")
print("="*70)

gini_results = []
best_gini_feature = None
best_gini_threshold = None
best_gini_gain = -np.inf

for i, feature_name in enumerate(feature_names):
    threshold, gain = find_best_split_for_feature(X[:, i], y, gini_index)
    gini_results.append({
        'feature': feature_name,
        'threshold': threshold,
        'gain': gain
    })
    
    print(f"\nFeature: {feature_name}")
    print(f"  Best Threshold: {threshold:.4f}" if threshold is not None else "  Best Threshold: None")
    print(f"  Information Gain: {gain:.6f}")
    
    if gain > best_gini_gain:
        best_gini_gain = gain
        best_gini_threshold = threshold
        best_gini_feature = feature_name

print("\n" + "="*70)
print(f"OVERALL BEST SPLIT (GINI):")
print(f"  Feature: {best_gini_feature}")
print(f"  Threshold: {best_gini_threshold:.4f}")
print(f"  Information Gain: {best_gini_gain:.6f}")
print("="*70)

BEST SPLITS USING GINI INDEX

Feature: Pregnancies
  Best Threshold: 6.5000
  Information Gain: 0.025642

Feature: Glucose
  Best Threshold: 127.5000
  Information Gain: 0.082500

Feature: BloodPressure
  Best Threshold: 69.0000
  Information Gain: 0.008713

Feature: SkinThickness
  Best Threshold: 31.5000
  Information Gain: 0.010883

Feature: Insulin
  Best Threshold: 121.0000
  Information Gain: 0.017369

Feature: BMI
  Best Threshold: 29.8500
  Information Gain: 0.042870

Feature: DiabetesPedigreeFunction
  Best Threshold: 0.5275
  Information Gain: 0.013310

Feature: Age
  Best Threshold: 28.5000
  Information Gain: 0.044259

OVERALL BEST SPLIT (GINI):
  Feature: Glucose
  Threshold: 127.5000
  Information Gain: 0.082500


## 6. Find Best Split for All Features Using Entropy

In [20]:
print("="*70)
print("BEST SPLITS USING ENTROPY")
print("="*70)

entropy_results = []
best_entropy_feature = None
best_entropy_threshold = None
best_entropy_gain = -np.inf

for i, feature_name in enumerate(feature_names):
    threshold, gain = find_best_split_for_feature(X[:, i], y, entropy)
    entropy_results.append({
        'feature': feature_name,
        'threshold': threshold,
        'gain': gain
    })
    
    print(f"\nFeature: {feature_name}")
    print(f"  Best Threshold: {threshold:.4f}" if threshold is not None else "  Best Threshold: None")
    print(f"  Information Gain: {gain:.6f}")
    
    if gain > best_entropy_gain:
        best_entropy_gain = gain
        best_entropy_threshold = threshold
        best_entropy_feature = feature_name

print("\n" + "="*70)
print(f"OVERALL BEST SPLIT (ENTROPY):")
print(f"  Feature: {best_entropy_feature}")
print(f"  Threshold: {best_entropy_threshold:.4f}")
print(f"  Information Gain: {best_entropy_gain:.6f}")
print("="*70)

BEST SPLITS USING ENTROPY

Feature: Pregnancies
  Best Threshold: 6.5000
  Information Gain: 0.039180

Feature: Glucose
  Best Threshold: 127.5000
  Information Gain: 0.130810

Feature: BloodPressure
  Best Threshold: 69.0000
  Information Gain: 0.014049

Feature: SkinThickness
  Best Threshold: 31.5000
  Information Gain: 0.016903

Feature: Insulin
  Best Threshold: 121.0000
  Information Gain: 0.026802

Feature: BMI
  Best Threshold: 27.8500
  Information Gain: 0.074899

Feature: DiabetesPedigreeFunction
  Best Threshold: 0.5275
  Information Gain: 0.020796

Feature: Age
  Best Threshold: 28.5000
  Information Gain: 0.072473

OVERALL BEST SPLIT (ENTROPY):
  Feature: Glucose
  Threshold: 127.5000
  Information Gain: 0.130810


## 7. Compare Gini Index vs Entropy Results

In [21]:
print("="*70)
print("COMPARISON: GINI INDEX vs ENTROPY")
print("="*70)

print("\n" + "-"*70)
print(f"{'Feature':<30} {'Gini Threshold':<18} {'Entropy Threshold':<18}")
print("-"*70)

for gini_res, entropy_res in zip(gini_results, entropy_results):
    gini_thresh = f"{gini_res['threshold']:.4f}" if gini_res['threshold'] is not None else "None"
    entropy_thresh = f"{entropy_res['threshold']:.4f}" if entropy_res['threshold'] is not None else "None"
    print(f"{gini_res['feature']:<30} {gini_thresh:<18} {entropy_thresh:<18}")

print("-"*70)
print(f"\nBest Split Comparison:")
print(f"  Gini:    {best_gini_feature} <= {best_gini_threshold:.4f}")
print(f"  Entropy: {best_entropy_feature} <= {best_entropy_threshold:.4f}")

if best_gini_feature == best_entropy_feature and abs(best_gini_threshold - best_entropy_threshold) < 1e-6:
    print(f"\nRESULT: Both Gini and Entropy select the SAME split")
else:
    print(f"\nRESULT: Gini and Entropy select DIFFERENT splits")

print("="*70)

COMPARISON: GINI INDEX vs ENTROPY

----------------------------------------------------------------------
Feature                        Gini Threshold     Entropy Threshold 
----------------------------------------------------------------------
Pregnancies                    6.5000             6.5000            
Glucose                        127.5000           127.5000          
BloodPressure                  69.0000            69.0000           
SkinThickness                  31.5000            31.5000           
Insulin                        121.0000           121.0000          
BMI                            29.8500            27.8500           
DiabetesPedigreeFunction       0.5275             0.5275            
Age                            28.5000            28.5000           
----------------------------------------------------------------------

Best Split Comparison:
  Gini:    Glucose <= 127.5000
  Entropy: Glucose <= 127.5000

RESULT: Both Gini and Entropy select the SAM

## 8. Summary Statistics for Best Split

In [ ]:
best_feature_idx = feature_names.index(best_gini_feature)

left_mask = X[:, best_feature_idx] <= best_gini_threshold
right_mask = ~left_mask

y_left = y[left_mask]
y_right = y[right_mask]

print("="*70)
print(f"SPLIT STATISTICS FOR: {best_gini_feature} <= {best_gini_threshold:.4f}")
print("="*70)

print(f"\nLeft child (<=):")
print(f"  Samples: {len(y_left)}")
print(f"  Class 0: {np.sum(y_left == 0)}")
print(f"  Class 1: {np.sum(y_left == 1)}")
print(f"  Gini: {gini_index(y_left):.4f}")
print(f"  Entropy: {entropy(y_left):.4f}")

print(f"\nRight child (>):")
print(f"  Samples: {len(y_right)}")
print(f"  Class 0: {np.sum(y_right == 0)}")
print(f"  Class 1: {np.sum(y_right == 1)}")
print(f"  Gini: {gini_index(y_right):.4f}")
print(f"  Entropy: {entropy(y_right):.4f}")

print("="*70)

## Analysis

Results and Documentation

Feature Search: The algorithm successfully checks all 8 numerical features.

Threshold Calculation: For each feature, we sort the unique values and calculate the midpoints between consecutive values to ensure that we test all mathematically relevant split points.

Comparison:
- Both Gini Index and Entropy selected the same overall best split for the root node: Glucose <= 127.5000
- For most features, both methods chose the same threshold, except for BMI (Gini: 29.85, Entropy: 27.85)
- Gini Index measures impurity based on probability of misclassification
- Entropy measures impurity based on information theory (bits of information needed to identify the class)
- Despite different formulas, both methods often lead to similar or identical splitting decisions
